# 20 Wan 2.2: moving shots from keyframes
Settings: GPU T4 x2, Internet on. Add the dataset `heritage-short-bundle`. Each clip is saved the moment it is made, and clips already saved are skipped, so a timeout loses nothing: just run it again. Download `/kaggle/working/shots/` and `notebook_log.jsonl`.

**Untested.** The TI2V-5B image-to-video call below follows the diffusers docs (`WanImageToVideoPipeline` with `image=`) and the model card, but has not been run.

In [ ]:
!pip install -q -U diffusers transformers accelerate imageio imageio-ffmpeg ftfy sentencepiece

In [ ]:
import json, time
from pathlib import Path
import torch
from diffusers import AutoencoderKLWan, WanImageToVideoPipeline
from diffusers.utils import export_to_video, load_image

BUNDLE = next(p.parent for p in Path("/kaggle/input").rglob("storyboard.json"))  # the dataset from `make kaggle-bundle`, wherever Kaggle mounts it
OUT = Path("/kaggle/working/shots"); OUT.mkdir(exist_ok=True)
LOG = Path("/kaggle/working/notebook_log.jsonl")
jobs = json.loads((BUNDLE / "wan_jobs.json").read_text())

MODEL = "Wan-AI/Wan2.2-TI2V-5B-Diffusers"
vae = AutoencoderKLWan.from_pretrained(MODEL, subfolder="vae", torch_dtype=torch.float32)
pipe = WanImageToVideoPipeline.from_pretrained(MODEL, vae=vae, torch_dtype=torch.float16)  # T4 has no bfloat16
pipe.enable_model_cpu_offload()

NEGATIVE = "blurry, low quality, distorted face, extra fingers, deformed, static, text, watermark"
for job in jobs:
    out = OUT / f"{job['id']}.mp4"
    if out.exists():
        print(job["id"], "already done"); continue
    image = load_image(str(BUNDLE / job["keyframe"])).resize((480, 832))
    start = time.time()
    frames = pipe(image=image, prompt=job["prompt"], negative_prompt=NEGATIVE, height=832, width=480,
                  num_frames=81, guidance_scale=5.0, num_inference_steps=30,
                  generator=torch.Generator("cpu").manual_seed(job["seed"])).frames[0]
    export_to_video(frames, str(out), fps=24)
    # the job hash lets the pipeline tell a stale clip (older keyframe, prompt or seed) from a current one
    (OUT / f"{job['id']}.json").write_text(json.dumps({"hash": job["hash"]}))
    seconds = time.time() - start
    with LOG.open("a") as f:
        f.write(json.dumps({"stage": "shots", "model": MODEL, "shot_id": job["id"], "seconds": round(seconds, 1),
                            "seed": job["seed"], "hash": job["hash"], "width": 480, "height": 832,
                            "frames": 81}) + "\n")
    print(job["id"], round(seconds), "s")